# MinGRU Phase 1.3 — Colab A100 training

**Goal:** Reach coherent TinyStories generation on the A100 in a reasonable
wall-clock budget (target ≤ 1 hour for the Phase 1.3 gate).

**Architecture:** `MinGRUModel(d=256, L=6, d_ffn=768, vocab=4000)` — 5.75M
params, scaffolded in `cubemind/training/vsa_lm.py` (Phase 1.2) and
trained by `sandbox/mingru_baseline/train.py`.

**Optimizer:** `grilly.optim.AdamW`, cosine LR decay from 3e-4 → 1e-5,
warmup 1000, `grad_clip=1.0` (per `TASKS.md` Phase 1.3).

**Runtime:** `Runtime → Change runtime type → GPU (A100)`. The notebook
checks `grilly` imports after install — if grilly can't attach to the
Vulkan ICD on Colab's NVIDIA runtime, it falls back to its NumPy path
(slow but correct).

**Prerequisites:**
- cubemind + grilly repos installed on Colab (install cell below).
- `data/tinystories_50k.json` available. Upload the file once or let
  the notebook fall back to the HuggingFace V2 split download.

---

## 0 · Runtime check

In [ ]:
!nvidia-smi || echo 'no GPU attached — switch to GPU runtime'
import platform, sys
print('python :', sys.version.split()[0])
print('platform:', platform.platform())

## 1 · Install grilly + cubemind

Two ways to provide the repos:

- **Clone from GitHub** (default; requires the repos to be public or that
  you have a `GIT_TOKEN` secret set).
- **Upload zips** (fallback): drag-drop `grilly.zip` and `cubemind.zip` into
  Colab's Files pane and use the commented-out cell below.

`grilly` ships a compiled `grilly_core` Vulkan backend. On Colab's
Linux NVIDIA runtime that means the build must provide a Linux `.so` (if
available) or fall back to the numpy path inside grilly. Either works —
the training loop calls `grilly.nn.autograd` / `grilly.optim` which
handle both paths transparently.

In [ ]:
%cd /content
# --- Clone (edit branch/urls as needed) -----------------------------------
GRILLY_REPO  = 'https://github.com/Grillcheese-AI/grilly.git'
CUBEMIND_REPO = 'https://github.com/Grillcheese-AI/cubemind.git'
!rm -rf grilly cubemind
!git clone --depth 1 $GRILLY_REPO
!git clone --depth 1 $CUBEMIND_REPO
!ls

In [ ]:
# Pip installs.
#   --no-deps on grilly because we install numpy/tokenizers explicitly
#   and don't want setuptools to pull pinned older versions.
!pip install -q --upgrade pip setuptools wheel
!pip install -q numpy tokenizers loguru click dependency-injector fastapi pydantic
!pip install -q --no-build-isolation ./grilly
!pip install -q --no-build-isolation ./cubemind
print('done')

In [ ]:
# Smoke-test imports. grilly prints GPU device info on first import; if
# Vulkan ICD isn't available, you'll see a numpy fallback note instead.
import grilly
from grilly import nn as gnn
from grilly.optim import AdamW
from grilly.nn.prefix_scan import prefix_scan_causal

print('grilly version  :', grilly.__version__ if hasattr(grilly, '__version__') else 'editable')
print('Vulkan available:', getattr(grilly, 'VULKAN_AVAILABLE', '?'))

## 2 · Verify the prefix-scan seq-len cap is lifted

Confirms the shader + binding rebuilt with the per-thread serial scan so
MinGRU can run at TinyStories seq=256+. If this fails with `seq_len > 32`,
the grilly install picked up an older release — reinstall from source.

In [ ]:
import numpy as np
from grilly.nn.prefix_scan import prefix_scan_causal

B, S, D = 2, 256, 64
rng = np.random.default_rng(0)
x = rng.standard_normal((B, S, D)).astype(np.float32) * 0.1
a = np.clip(0.05 + rng.random((B, S, D)).astype(np.float32) * 0.9, 0.05, 0.95)
h = prefix_scan_causal(x, a)

# Reference: numpy scalar scan
h_ref = np.zeros_like(x); hp = np.zeros((B, D), dtype=np.float32)
for t in range(S):
    hp = a[:, t] * hp + x[:, t]
    h_ref[:, t] = hp
print(f'seq=256 max abs err vs numpy: {float(np.abs(h.data - h_ref).max()):.3e}')

## 3 · Dataset — upload `tinystories_50k.json`

The training script prefers the local 50k-story JSON (same file used in
`cubemind/data/tinystories_50k.json`). Drop it into the Files pane at
`/content/cubemind/data/tinystories_50k.json`, OR uncomment the
HuggingFace fallback cell.

In [ ]:
import os
from pathlib import Path

target = Path('/content/cubemind/data/tinystories_50k.json')
target.parent.mkdir(parents=True, exist_ok=True)

if not target.exists():
    # Prompt for upload via Colab's file picker.
    try:
        from google.colab import files
        print('Choose tinystories_50k.json to upload (~45 MB):')
        uploaded = files.upload()
        for name, data in uploaded.items():
            if 'tinystories' in name.lower():
                target.write_bytes(data)
                break
    except Exception as e:
        print(f'Upload failed ({e}) — falling back to HuggingFace split in train.py')

if target.exists():
    print(f'dataset ready: {target} ({target.stat().st_size/1e6:.1f} MB)')
else:
    print('no local JSON — prepare_data() will auto-download TinyStoriesV2 GPT-4')

## 4 · Train — full Phase 1.3 run

Args match TASKS.md 1.3 defaults (d=256, L=6, d_ffn=768, vocab=4000,
lr=3e-4, warmup=1000, clip=1.0). Adjust `--steps` / `--minutes` to your
time budget.

On A100 expected throughput is ~20-50× the RX 6750 XT's ~1000 tok/s. A
20K-step run typically lands in under an hour.

In [ ]:
%cd /content/cubemind
# Unbuffered output so Colab streams the training log live.
import os
os.environ['PYTHONUNBUFFERED'] = '1'

# Recommended defaults — tweak STEPS / EVAL_EVERY / LOG_EVERY as desired.
STEPS       = 20_000
MINUTES_CAP = 90
LOG_EVERY   = 50
EVAL_EVERY  = 2_000
CKPT_EVERY  = 1_000
BATCH_SIZE  = 16

!python -u sandbox/mingru_baseline/train.py \
    --steps $STEPS --minutes $MINUTES_CAP \
    --log-every $LOG_EVERY --eval-every $EVAL_EVERY --ckpt-every $CKPT_EVERY \
    --batch-size $BATCH_SIZE

## 5 · Inspect generations

`train.py` writes `gen_step_<N>.md` every `--eval-every` steps plus a
`generations_final.md` at the end. Quick peek at the latest.

In [ ]:
from pathlib import Path
results = Path('/content/cubemind/sandbox/mingru_baseline/results')
gens = sorted(results.glob('gen_step_*.md'))
final = results / 'generations_final.md'
print('artefacts:')
for p in gens + ([final] if final.exists() else []):
    print(f'  {p.name}  ({p.stat().st_size/1024:.1f} KB)')
if final.exists():
    print('\n--- generations_final.md (first 4 KB) ---\n')
    print(final.read_text(encoding='utf-8')[:4000])
elif gens:
    latest = gens[-1]
    print(f'\n--- {latest.name} (first 4 KB) ---\n')
    print(latest.read_text(encoding='utf-8')[:4000])

## 6 · Download results

Zip up `results/` (checkpoints + generations + summary.json) and pull it
back to your machine. Commit to `sandbox/mingru_baseline/results/` in
the cubemind repo.

In [ ]:
import shutil
from pathlib import Path
results = Path('/content/cubemind/sandbox/mingru_baseline/results')
assert results.exists(), 'nothing to download — training did not produce outputs'
out = shutil.make_archive('/content/mingru_baseline_results', 'zip', str(results))
print(f'packaged: {out}  ({Path(out).stat().st_size/1e6:.1f} MB)')

from google.colab import files
files.download(out)

## Post-run checklist

1. Extract `mingru_baseline_results.zip` into `sandbox/mingru_baseline/results/`
   in the local repo.
2. Review `generations_final.md` — visual coherence check per Phase 1.4.
3. If stories are grammatically correct and on-prompt, mark Phase 1.3 done
   in `TASKS.md`. If not, iterate: more steps, adjust seq_len, or revisit
   the architecture.
4. Phase 1.4 runs `eval_coherence.py` against `results/best.npz` for the
   GPT-4 rubric scores.